# 🛰️ Notebook 3: Calculating LULC 
This notebook introduces various ways to make **LULC** from STAC Data 

---

Let's start by searching for 3 months data over Auckland with cloud cover less than 10%

In [ ]:
from pystac_client import Client

# Connect to a public catalog
catalog = Client.open('https://earth-search.aws.element84.com/v1')

# Intersects
area_of_interest =  {
        "coordinates": [
          [
            [
              73.66059124581909,
              20.04371590757539
            ],
            [
              73.66059124581909,
              19.981360740605894
            ],
            [
              73.72993716518059,
              19.981360740605894
            ],
            [
              73.72993716518059,
              20.04371590757539
            ],
            [
              73.66059124581909,
              20.04371590757539
            ]
          ]
        ],
        "type": "Polygon"
      }

time_range = "2024-01-01/2024-03-31"

search = catalog.search(
    collections=["sentinel-2-l2a"], 
    query={"eo:cloud_cover": {"lt": 10}},
    intersects=area_of_interest, 
    datetime=time_range
)
items = list(search.get_items())
print(f'Found {len(items)} items')

/opt/homebrew/Caskroom/miniconda/base/envs/stac/lib/python3.12/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Found 28 items


Let's load this data using `odc-stac` 

In [2]:
from odc.stac import load
print('odc.stac available')


odc.stac available


In [ ]:
ds = load(items, 
          groupby='solar_day', # Controls what items are placed in similar pixel plane
          resolution=10,
          chunks={'x':1024,'y':1024}, # Optional : Helpful to divide data in chunks for calculation
          geopolygon=area_of_interest,
          )
ds

<xarray.Dataset> Size: 876MB
Dimensions:       (y: 697, x: 732, time: 26)
Coordinates:
  * y             (y) float64 6kB 2.217e+06 2.217e+06 ... 2.21e+06 2.21e+06
  * x             (x) float64 6kB 3.599e+05 3.599e+05 ... 3.672e+05 3.672e+05
  * time          (time) datetime64[ns] 208B 2024-01-02T05:53:23.226000 ... 2...
    spatial_ref   int32 4B 32643
Data variables: (12/32)
    aot           (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    blue          (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    coastal       (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    green         (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    nir           (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    nir08         (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    ...            ...
    rededge3-jp2  (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    scl-jp2       (time, y, x) uint8 13MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    swir16-jp2    (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    swir22-jp2    (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    visual-jp2    (time, y, x) float32 53MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>
    wvp-jp2       (time, y, x) uint16 27MB dask.array<chunksize=(1, 697, 732), meta=np.ndarray>

In [8]:
ds_mosaic = ds.mean(dim='time')
type(ds_mosaic)

xarray.core.dataset.Dataset

Calculating `NDVI` and `NDWI` so we can make rules about LULC classification

In [9]:
ndvi = (ds_mosaic.nir - ds_mosaic.red) / (ds_mosaic.nir + ds_mosaic.red)
ndwi = (ds_mosaic.green - ds_mosaic.nir) / (ds_mosaic.green + ds_mosaic.nir)

In [10]:
import xarray as xr

lulc = xr.full_like(ndvi, fill_value=0, dtype="uint8")

# WATER
lulc = lulc.where(~((ndwi > 0.3) & (ndvi < 0.1)), 1)

# BUILT-UP
lulc = lulc.where(~((ndwi < 0.1) & (ndvi < 0.2)), 2)

# BARE SOIL
lulc = lulc.where(~((ndvi > 0.1) & (ndvi < 0.2) & (ndwi < 0)), 3)

# VEGETATION
lulc = lulc.where(~((ndvi > 0.2) & (ndvi < 0.4)), 4)

# DENSE VEGETATION
lulc = lulc.where(~(ndvi > 0.4), 5)

# WETLANDS
lulc = lulc.where(~((ndwi > 0.1) & (ndwi < 0.3) & (ndvi > 0.2)), 6)


In [15]:
import rioxarray
lulc = lulc.rio.write_crs(ds_mosaic.rio.crs)
lulc = lulc.rio.write_transform(ds_mosaic.rio.transform())
lulc.rio.to_raster("lulc_classification.tif", dtype="uint8")

